[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/skarma91/logicmojo-ai-july-2026/blob/main/modules/module-3-nlp-to-transformers/milestone-assignment/solution/solution.ipynb)

# Module 3 milestone: reference solution

The text-intelligence layer for a support inbox, built stage by stage. Each code cell prints its result and notes the expected output in a comment. Needs internet for the first model download.

## Setup

New libraries for this notebook (PyTorch already came with class 2.4, PyTorch fundamentals):

```
pip install transformers sentence-transformers
```

## The scenario and the fixed dataset

Eight support messages for a SaaS accounting product, the four teams the inbox routes to, and a six-entry help centre. Everything downstream runs on these fixed inputs.

In [ ]:
messages = [
    "I was charged twice for my subscription this month and I want a refund.",
    "My card was declined at checkout but I was still billed for the upgrade.",
    "The app crashes every time I open the bank reconciliation screen.",
    "After the latest update my reports show the wrong totals, this is broken.",
    "How do I export my invoices to a CSV file?",
    "Where is the setting to add a second user to my account?",
    "Please cancel my subscription, I do not need the service any more.",
    "I want to close my account and stop all future payments.",
]

teams = {
    "Billing":      "A question or complaint about charges, payments, refunds, invoices, or subscription fees.",
    "Bug report":   "A report that the software is broken, crashing, showing errors, or giving wrong results.",
    "How-to":       "A question about how to use a feature or where to find a setting in the product.",
    "Cancellation": "A request to cancel the subscription or close the account.",
}

faqs = [
    "To export invoices, open Sales, then Invoices, select the rows, and choose Export to CSV.",
    "To add a user, open Settings, then Manage Users, click Add User, and set their role.",
    "To change your payment method, open Billing, then Payment Methods, and edit the card on file.",
    "To cancel your plan, open Billing, then Subscription, and choose Cancel Subscription.",
    "To reconcile an account, open Accounting, then Reconcile, and pick the account and statement date.",
    "If report totals look wrong, open Reports, then Settings, and refresh the data or clear the cache.",
]

for i, m in enumerate(messages, 1):
    print(i, m)

### Part 1. Tokenization audit (class 3.1, Text as data)

How does a real tokenizer see these messages? WordPiece keeps common words whole and breaks rarer ones into subword pieces marked with `##`.

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("bert-base-uncased")

print(f"{'#':>2}  {'words':>5}  {'tokens':>6}   pieces")
for i, m in enumerate(messages, 1):
    pieces = tok.tokenize(m)
    print(f"{i:>2}  {len(m.split()):>5}  {len(pieces):>6}   {pieces}")

# Expected: for every message the token count is at or above the word count,
# because rarer words split into several ## pieces. Exact pieces depend on the
# vocab, but you will see something like:
#   'reconciliation' -> several ## pieces, 'invoices' -> 2 to 3 pieces,
#   'csv' -> 2 pieces, while 'charged', 'refund', 'account' stay whole.

In [ ]:
# Which domain words split the hardest?
for w in ["reconciliation", "invoices", "subscription", "csv", "checkout", "refund"]:
    print(f"{w:>15}  ->  {tok.tokenize(w)}")

# Takeaway: a transformer is billed and budgeted in TOKENS, not words. A message
# that looks short in words can be noticeably longer in tokens once domain terms
# and punctuation are split, and those tokens are what spend the context window.

### Part 2. Sentiment triage (class 3.4, Using pretrained transformers)

Flag unhappy customers so they surface first. One pipeline call scores every message.

In [ ]:
from transformers import pipeline

clf = pipeline("sentiment-analysis")
scored = clf(messages)

# priority = a confident NEGATIVE; sort so the angriest customers come first
def priority(r):
    return r["score"] if r["label"] == "NEGATIVE" else 0.0

ranked = sorted(zip(messages, scored), key=lambda pair: -priority(pair[1]))
for m, r in ranked:
    flag = "HIGH" if (r["label"] == "NEGATIVE" and r["score"] > 0.9) else "    "
    print(f"[{flag}] {r['label']:8} {r['score']:.2f}  {m}")

# Expected: the billing complaints and the two bug reports come back NEGATIVE at
# ~0.99 and are flagged HIGH; the cancellations usually read NEGATIVE as well.
# The two how-to questions are neutral, but this model was trained on SST-2, which
# has only POSITIVE and NEGATIVE. It is forced to pick a side, often at lower
# confidence. That is the practical limit of a binary sentiment model.

A neutral question has no good answer in a two-class model. In production you would either use a three-class model or treat low-confidence predictions as neutral. Naming that limit is part of the exercise.

### Part 3. Routing and FAQ retrieval with embeddings (classes 3.2 Attention, 3.3 The Transformer, 3.4)

First the mechanic you learned: turn a sentence into one vector by mean-pooling an encoder's `last_hidden_state`. Then the real task: route each message to a team and pull the best help article.

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

name = "distilbert-base-uncased"
d_tok = AutoTokenizer.from_pretrained(name)
d_model = AutoModel.from_pretrained(name)
d_model.eval()

def embed_meanpool(text):
    enc = d_tok(text, return_tensors="pt")
    with torch.no_grad():
        hidden = d_model(**enc).last_hidden_state          # [1, seq, 768]
    mask = enc["attention_mask"].unsqueeze(-1).float()     # [1, seq, 1]
    return (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)   # [1, 768]

cos = lambda x, y: float(F.cosine_similarity(x, y))
a = embed_meanpool("How do I export my invoices to a CSV file?")
b = embed_meanpool("What is the way to save my invoices as a CSV?")   # paraphrase of a
z = embed_meanpool("The bank reconciliation screen keeps crashing.")  # unrelated
print("paraphrase :", round(cos(a, b), 3))
print("unrelated  :", round(cos(a, z), 3))

# Expected: the paraphrase scores higher than the unrelated sentence. Note the gap
# is small: a raw base model squeezes everything into a narrow similarity band.
# That is exactly why a model tuned for similarity does the routing below better.

In [ ]:
# A purpose-built sentence encoder, then route each message to the nearest team.
from sentence_transformers import SentenceTransformer, util

st = SentenceTransformer("all-MiniLM-L6-v2")   # small, tuned for semantic similarity
team_names = list(teams)
team_desc = list(teams.values())

msg_emb = st.encode(messages, convert_to_tensor=True, normalize_embeddings=True)
team_emb = st.encode(team_desc, convert_to_tensor=True, normalize_embeddings=True)
sims = util.cos_sim(msg_emb, team_emb)         # [8 messages, 4 teams]

for i, m in enumerate(messages, 1):
    j = int(sims[i - 1].argmax())
    print(f"{i}  {team_names[j]:12}  {m}")

# Expected routing:
#   1 Billing        2 Billing
#   3 Bug report     4 Bug report
#   5 How-to         6 How-to
#   7 Cancellation   8 Cancellation

In [ ]:
# For the two how-to questions, retrieve the single best FAQ answer.
faq_emb = st.encode(faqs, convert_to_tensor=True, normalize_embeddings=True)
for i in (5, 6):
    q = st.encode(messages[i - 1], convert_to_tensor=True, normalize_embeddings=True)
    j = int(util.cos_sim(q, faq_emb).argmax())
    print(f"msg {i}:  {messages[i - 1]}")
    print(f"   -> {faqs[j]}\n")

# Expected:
#   msg 5 -> the "export invoices ... Export to CSV" FAQ
#   msg 6 -> the "add a user ... Manage Users" FAQ

### Part 4. Draft a reply and control the decoding (class 3.5, From models to LLMs)

Same prompt, three decoding settings. Watch how greedy, temperature, and top-p change the draft.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, set_seed
import torch

g_tok = AutoTokenizer.from_pretrained("distilgpt2")
g_model = AutoModelForCausalLM.from_pretrained("distilgpt2")
g_model.eval()

prompt = ("Customer question: How do I export my invoices to a CSV file?\n"
          "Support reply:")
ids = g_tok(prompt, return_tensors="pt")
n = ids["input_ids"].shape[1]
window = g_model.config.n_positions
print(f"prompt tokens: {n} / context window {window}  ({100 * n / window:.1f}%)")

def draft(**kw):
    set_seed(0)                       # reproducible sampling
    with torch.no_grad():
        out = g_model.generate(**ids, max_new_tokens=40,
                               pad_token_id=g_tok.eos_token_id, **kw)
    return g_tok.decode(out[0][n:], skip_special_tokens=True).strip()

print("\nGREEDY    :", draft(do_sample=False))
print("\nTEMP 1.3  :", draft(do_sample=True, temperature=1.3, top_k=0))
print("\nTOP-P 0.9 :", draft(do_sample=True, top_p=0.9, top_k=0))

# Expected: the prompt is a small fraction of the 1024-token window. Greedy is
# deterministic and tends to repeat a phrase; temperature 1.3 is more varied and
# can wander off topic; top-p 0.9 sits in between, fluent but controlled. The
# three drafts read differently. distilgpt2 is tiny, so the wording is rough:
# you are studying the decoding knobs, not the quality of a small model.

### Write-up (reference)

Tokenization set the scale of everything: WordPiece split domain words like "reconciliation" and "invoices" into several `##` pieces, so each message cost more tokens than it had words, and tokens are what a model is billed and budgeted in. Attention is what let the later stages read meaning rather than keywords. In sentiment, self-attention lets "not" attach to "recommend" and "charged twice" attach to "refund", so the model scores intent rather than counting positive and negative words. In routing, the same contextual mixing is baked into each sentence embedding, so "my card was declined but I was still billed" lands near the Billing description even though it shares few exact words with it. The base DistilBERT vectors showed the idea but compressed everything into a narrow band, which is why the purpose-built encoder routed the messages cleanly while the raw model barely separated a paraphrase from an unrelated sentence.